# Document Generation

> Notebook generated from [`examples/subgraphs/07_document_generation.py`](../../examples/subgraphs/07_document_generation.py).

| Item | Value |
|------|-------|
| **Dataset** | Wikipedia Technical Docs (embedded) |
| **API key** | ⚠️  **Requires API key** (`ANTHROPIC_API_KEY` or `OPENAI_API_KEY`) in `.env`. |

**Expected result:** planner → researcher → writer → editor → formatter. Final document in markdown, html and plain.

---

## Setup

```bash
uv pip install -e ".[dev,all]"
```

> To use `await main()` directly in cells, this notebook
> installs `nest_asyncio` in the first cell.

---

## Original docstring

```
Document Generation Subgraph — Technical document generation
================================================================
Subgraph: prismal.agents.subgraphs.document_generation

Dataset: Wikipedia Technical Docs + OpenAPI Specifications
  • Topics: REST API Design, Machine Learning Explainability, Zero-Trust Security
  • Reference: https://en.wikipedia.org/wiki/REST and XAI papers
  • Why: The document_generation subgraph has 5 nodes in a linear pipeline
    (planner → researcher → writer → editor → formatter). Technical topics
    have predictable structure (introduction, key concepts, examples,
    best practices, references) that lets us validate each node of the pipeline.
    Each node transforms the document incrementally, making the value
    of each stage visible.

Document Generation subgraph description:
  planner → researcher → writer → editor → formatter

  Nodes:
  1. planner    — defines the document structure (sections, audience, length)
  2. researcher — collects relevant information (LLM + optional RAG)
  3. writer     — drafts the complete document following the plan
  4. editor     — reviews clarity, coherence, technical consistency
  5. formatter  — applies final format (markdown/plain/html) and metadata

Usage:
    uv run python examples/subgraphs/07_document_generation.py
```

In [ ]:
# Enable nested event loop to use `await` directly in Jupyter.
import sys
if 'nest_asyncio' not in sys.modules:
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        %pip install -q nest_asyncio
        import nest_asyncio
        nest_asyncio.apply()

## Setup · imports and dataset

In [ ]:
from __future__ import annotations

import asyncio

# Import with error handling
try:
    from prismal.agents.subgraphs.document_generation.builder import (
        build_document_generation_subgraph,
        register_document_generation,
    )
    from prismal.agents.subgraphs.factory import assemble_state_graph

    DOC_GEN_AVAILABLE = True
except ImportError:
    DOC_GEN_AVAILABLE = False

## Dataset: technical document requests

In [ ]:
DOCUMENT_REQUESTS = [
    {
        "id": "DOC-001",
        "title": "REST API Design Guide",
        "topic": "REST API design best practices",
        "audience": "developers intermediate",
        "format": "markdown",
        "sections": [
            "Introduction",
            "Resources and URIs",
            "HTTP Methods",
            "Status Codes",
            "Versioning",
            "Authentication",
            "Pagination",
            "Best Practices",
        ],
        "target_length": "1500 words",
        "context": (
            "Practical guide for backend developers who want to design robust "
            "and maintainable REST APIs. Must include examples of well- and poorly-designed URIs."
        ),
    },
    {
        "id": "DOC-002",
        "title": "Introduction to ML Explainability (XAI)",
        "topic": "Machine Learning Explainability (SHAP, LIME, feature importance)",
        "audience": "data scientists intermediate",
        "format": "markdown",
        "sections": [
            "Why XAI?",
            "SHAP Values",
            "LIME",
            "Feature Importance",
            "Use Cases",
            "Limitations",
            "Tools",
        ],
        "target_length": "1200 words",
        "context": (
            "Document for data scientists who train production models and "
            "need to explain their predictions to non-technical stakeholders."
        ),
    },
    {
        "id": "DOC-003",
        "title": "Zero-Trust Security Architecture",
        "topic": "Zero-trust network security model",
        "audience": "security engineers advanced",
        "format": "markdown",
        "sections": [
            "Core Principles",
            "Identity Verification",
            "Least Privilege",
            "Microsegmentation",
            "Continuous Monitoring",
            "Implementation Roadmap",
        ],
        "target_length": "2000 words",
        "context": (
            "Technical reference for security engineers migrating from perimeter-based "
            "to zero-trust architecture. Includes practical implementation steps."
        ),
    },
]

## Base knowledge for each topic (simulates what the researcher extracts)

In [ ]:
RESEARCH_KNOWLEDGE = {
    "REST API design best practices": {
        "key_concepts": [
            "Resources are nouns, not verbs: /users not /getUsers",
            "Use HTTP methods semantically: GET (read), POST (create), PUT/PATCH (update), DELETE",
            "HTTP status codes: 200 OK, 201 Created, 400 Bad Request, 401 Unauthorized, "
            "404 Not Found, 422 Unprocessable Entity, 500 Internal Server Error",
            "Versioning strategies: URI (/v1/users), Header (Accept: application/vnd.api.v1+json), "
            "Query param (?version=1)",
            "Pagination: cursor-based (more scalable) vs offset-based (simpler)",
            "Authentication: OAuth 2.0 + JWT for stateless auth",
            "HATEOAS: Hypermedia as the Engine of Application State",
        ],
        "good_examples": [
            "GET /users → list users",
            "GET /users/123 → get user 123",
            "POST /users → create user",
            "PATCH /users/123 → update user 123",
            "DELETE /users/123 → delete user 123",
            "GET /users/123/orders → user's orders",
        ],
        "bad_examples": [
            "GET /getUsers ← verb in URI",
            "POST /deleteUser ← wrong method",
            "GET /users?action=delete ← action as query param",
        ],
        "references": ["Roy Fielding's REST dissertation (2000)", "RFC 7230-7235", "OpenAPI 3.1"],
    },
    "Machine Learning Explainability (SHAP, LIME, feature importance)": {
        "key_concepts": [
            "SHAP (SHapley Additive exPlanations): game-theory based, global + local explanations",
            "LIME (Local Interpretable Model-agnostic Explanations): local surrogate models",
            "Feature importance: permutation importance (model-agnostic), gain importance (tree-based)",
            "Global explainability: understand model overall behavior",
            "Local explainability: explain single prediction",
            "GDPR Article 22: right to explanation for automated decisions",
        ],
        "formulas": [
            "SHAP value φᵢ = Σ [|S|!(|F|-|S|-1)!/|F|!] × [f(S∪{i}) - f(S)]",
            "LIME: arg min L(f, g, πₓ) + Ω(g)",
        ],
        "use_cases": [
            "Credit scoring: why was a loan rejected?",
            "Medical diagnosis: which features drove the prediction?",
            "Fraud detection: explain flagged transaction",
        ],
        "tools": ["shap library", "lime library", "eli5", "Captum (PyTorch)", "InterpretML"],
    },
    "Zero-trust network security model": {
        "key_concepts": [
            "Never trust, always verify — no implicit trust based on network location",
            "Verify explicitly: authenticate and authorize every request",
            "Use least privilege access: minimal necessary permissions",
            "Assume breach: design as if the network is already compromised",
            "Microsegmentation: divide network into small zones",
            "Continuous monitoring: log, inspect, and analyze all traffic",
        ],
        "principles": [
            "Identity is the new perimeter",
            "Device health verification",
            "Application-layer encryption (mTLS)",
            "Just-in-time (JIT) access provisioning",
        ],
        "frameworks": ["NIST SP 800-207", "BeyondCorp (Google)", "Forrester ZTX", "CISA ZTA"],
        "implementation_steps": [
            "1. Identify sensitive data and workflows",
            "2. Map transaction flows",
            "3. Build a Zero Trust architecture",
            "4. Create Zero Trust policy",
            "5. Monitor and maintain",
        ],
    },
}

## Pipeline simulator

In [ ]:
def simulate_planner(request: dict) -> dict:
    """Simulate the planner node: generates the document plan."""
    return {
        "title": request["title"],
        "audience": request["audience"],
        "sections": request["sections"],
        "estimated_words_per_section": int(request["target_length"].split()[0])
        // len(request["sections"]),
        "format": request["format"],
        "tone": "technical" if "advanced" in request["audience"] else "approachable-technical",
    }


def simulate_researcher(request: dict, plan: dict) -> dict:
    """Simulate the researcher node: collects relevant information."""
    kb = RESEARCH_KNOWLEDGE.get(request["topic"], {})
    return {
        "key_concepts": kb.get("key_concepts", []),
        "examples": kb.get("good_examples", []) + kb.get("bad_examples", []),
        "formulas": kb.get("formulas", []),
        "use_cases": kb.get("use_cases", []),
        "tools": kb.get("tools", []),
        "frameworks": kb.get("frameworks", []),
        "references": kb.get("references", []),
        "sources_count": len(kb),
    }


def simulate_writer(request: dict, plan: dict, research: dict) -> str:
    """Simulate the writer node: drafts the document."""
    sections = plan["sections"]
    concepts = research.get("key_concepts", [])
    examples = research.get("examples", [])

    doc = f"# {request['title']}\n\n"
    doc += f"*Audience: {plan['audience']} | Format: {plan['format']}*\n\n"

    for i, section in enumerate(sections):
        doc += f"## {section}\n\n"
        # Add simulated content per section
        if i < len(concepts):
            doc += f"{concepts[i]}\n\n"
        if i == 0:
            doc += f"{request['context']}\n\n"
        if examples and i == len(sections) // 2:
            doc += "**Examples:**\n"
            for ex in examples[:3]:
                doc += f"- `{ex}`\n"
            doc += "\n"

    if research.get("references"):
        doc += "## References\n\n"
        for ref in research["references"]:
            doc += f"- {ref}\n"

    return doc


def simulate_editor(draft: str, request: dict) -> tuple[str, list[str]]:
    """Simulate the editor node: review and improve the draft."""
    edits = []
    edited = draft

    # Check that there are H2 sections
    h2_count = draft.count("\n## ")
    if h2_count < len(request["sections"]) - 1:
        edits.append(f"Added {len(request['sections']) - h2_count} missing sections")

    # Check minimum length
    word_count = len(draft.split())
    target = int(request["target_length"].split()[0])
    if word_count < target * 0.5:
        edits.append(f"Document too short ({word_count} words vs {target} target)")

    # Add conclusion note if missing
    if "Conclusion" not in draft:
        edited += "\n## Conclusion\n\nThis document covers the fundamental aspects of the topic. "
        edited += "It is recommended to complement it with the official referenced documentation.\n"
        edits.append("Added Conclusion section")

    # Check tone consistency
    if request["audience"].endswith("advanced") and "?" in draft[:200]:
        edits.append("Tone adjusted: more technical for advanced audience")

    return edited, edits


def simulate_formatter(document: str, format_type: str) -> str:
    """Simulate the formatter node: applies the final format."""
    if format_type == "markdown":
        # Already Markdown, add YAML frontmatter metadata
        frontmatter = "---\nformat: markdown\ngenerator: prismal-document-generation\n---\n\n"
        return frontmatter + document
    if format_type == "html":
        # Basic HTML conversion
        html = "<html><body>\n"
        for line in document.splitlines():
            if line.startswith("# "):
                html += f"<h1>{line[2:]}</h1>\n"
            elif line.startswith("## "):
                html += f"<h2>{line[3:]}</h2>\n"
            elif line.startswith("- "):
                html += f"<li>{line[2:]}</li>\n"
            elif line.strip():
                html += f"<p>{line}</p>\n"
        html += "</body></html>"
        return html
    # plain
    import re

    return re.sub(r"[#*`_]", "", document)


async def run_document_generation(request: dict) -> dict:
    """Run the document generation pipeline."""
    print(f"\n[{request['id']}] {request['title']}")
    print(f"  Audience: {request['audience']} | Format: {request['format']}")
    print(f"  Target  : {request['target_length']}")

    if not DOC_GEN_AVAILABLE:
        print("  [Demo mode — simulated subgraph]")

        # Node 1: planner
        print("\n  ── Node 1: planner ──")
        plan = simulate_planner(request)
        print(f"    Planned sections      : {len(plan['sections'])}")
        print(f"    ~{plan['estimated_words_per_section']} words/section")
        print(f"    Tone: {plan['tone']}")

        # Node 2: researcher
        print("\n  ── Node 2: researcher ──")
        research = simulate_researcher(request, plan)
        print(f"    Key concepts found         : {len(research.get('key_concepts', []))}")
        print(f"    Examples gathered          : {len(research.get('examples', []))}")
        if research.get("references"):
            print(f"    Referenced sources         : {research['references']}")

        # Node 3: writer
        print("\n  ── Node 3: writer ──")
        draft = simulate_writer(request, plan, research)
        word_count = len(draft.split())
        print(f"    Draft generated  : {word_count} words, {len(draft)} chars")
        print(f"    H2 sections      : {draft.count(chr(10) + '## ')}")

        # Node 4: editor
        print("\n  ── Node 4: editor ──")
        final_doc, edits = simulate_editor(draft, request)
        if edits:
            print(f"    Edits applied ({len(edits)}):")
            for edit in edits:
                print(f"      • {edit}")
        else:
            print("    ✓ No edits needed")
        print(f"    Final word count: {len(final_doc.split())}")

        # Node 5: formatter
        print("\n  ── Node 5: formatter ──")
        formatted_doc = simulate_formatter(final_doc, request["format"])
        print(f"    Applied format    : {request['format']}")
        print(f"    Final document    : {len(formatted_doc)} chars")

        # Document preview
        lines = formatted_doc.splitlines()
        preview_lines = [l for l in lines[:12] if l.strip()][:6]
        print("\n  Preview (first lines):")
        for line in preview_lines:
            print(f"    {line[:75]}")

        return {
            "id": request["id"],
            "title": request["title"],
            "word_count": len(final_doc.split()),
            "sections": plan["sections"],
            "edits_applied": len(edits),
            "format": request["format"],
            "document": formatted_doc,
        }

    # Real mode with LangGraph subgraph
    from langchain_core.messages import HumanMessage

    from prismal.agents.state import create_initial_state

    await register_document_generation(format=request["format"])
    subgraph = build_document_generation_subgraph(format=request["format"])
    graph = assemble_state_graph(subgraph).compile()

    state = create_initial_state(session_id="nb-document-generation")
    state["messages"] = [
        HumanMessage(
            content=(
                f"Generate a technical document about: {request['topic']}\n"
                f"Title: {request['title']}\n"
                f"Audience: {request['audience']}\n"
                f"Sections: {', '.join(request['sections'])}\n"
                f"Target length: {request['target_length']}\n"
                f"Context: {request['context']}"
            )
        )
    ]
    state["metadata"] = {
        "document_generation": {
            "title": request["title"],
            "topic": request["topic"],
            "audience": request["audience"],
            "sections": request["sections"],
        }
    }

    config = {"configurable": {"thread_id": f"docgen_{request['id']}_001"}}
    final_state = await graph.ainvoke(state, config=config)

    messages = final_state.get("messages", [])
    doc_meta = final_state.get("metadata", {}).get("document_generation", {})
    return {
        "id": request["id"],
        "title": request["title"],
        "document": str(messages[-1].content) if messages else "",
        "word_count": doc_meta.get("word_count", 0),
    }


async def main() -> None:
    print("=" * 70)
    print("  Document Generation Subgraph — Dataset: Wikipedia Technical Docs")
    print("=" * 70)

    print("\n[Document Generation subgraph architecture]")
    pipeline = [
        ("planner   ", "Defines sections, audience, length, tone"),
        ("researcher", "Collects information: LLM knowledge + RAG (optional)"),
        ("writer    ", "Drafts the complete document following the plan"),
        ("editor    ", "Reviews coherence, clarity, technical completeness"),
        ("formatter ", "Applies the final format: markdown / plain / html"),
    ]
    for node, desc in pipeline:
        print(f"  {node}: {desc}")
    print()
    print("  Each node reads/writes metadata['document_generation']")
    print("  The formatter also appends an AIMessage with the final document")

    print(f"\n[Generating {len(DOCUMENT_REQUESTS)} technical documents]")
    results = []
    for request in DOCUMENT_REQUESTS:
        result = await run_document_generation(request)
        results.append(result)
        print("─" * 70)

    # ── Statistics ────────────────────────────────────────────────────────────
    print("\n[Summary of generated documents]")
    print(f"  {'ID':<10} {'Title':<35} {'Words':>8} {'Edits':>6} {'Format'}")
    print("  " + "─" * 68)
    for r in results:
        print(
            f"  {r['id']:<10} {r['title'][:34]:<35} {r['word_count']:>8} "
            f"{r['edits_applied']:>6} {r['format']}"
        )

    total_words = sum(r["word_count"] for r in results)
    print(f"\n  Total words generated      : {total_words:,}")
    print(f"  Average per document       : {total_words // len(results):,}")

    # ── Available formats ─────────────────────────────────────────────────────
    print("\n[Available output formats]")
    formats_info = [
        ("markdown", "Text with Markdown syntax — ideal for wikis, GitHub, Confluence"),
        ("plain", "Plain text without markup — useful for email, TTS, legacy systems"),
        ("html", "Renderable HTML — for web portals or rich emails"),
    ]
    for fmt, desc in formats_info:
        print(f"  {fmt:10s}: {desc}")

    print("\n[Document Generation Subgraph use cases]")
    use_cases = [
        "Automatic API technical documentation (from OpenAPI spec)",
        "Generation of periodic reports (weekly/monthly summaries)",
        "Writing of RFCs and ADRs (Architecture Decision Records)",
        "Creation of user manuals from feature specs",
        "Generation of content marketing (technical blogs)",
        "Code documentation (README, CONTRIBUTING, CHANGELOG)",
    ]
    for uc in use_cases:
        print(f"  ✓ {uc}")

    print("\n[RAG integration]")
    print("  subgraph = build_document_generation_subgraph(")
    print("      rag_engine=ChromaVectorStore(collection_name='company_docs'),")
    print("      format='markdown',")
    print("  )")
    print("  # The researcher will use RAG to ground the content")
    print("  # in the company's internal documents")


if __name__ == "__main__":
    asyncio.run(main())

---

## Run the demo

The original file ends with `asyncio.run(main())`. In Jupyter use:

```python
await main()
```

Thanks to the `nest_asyncio.apply()` in the first cell, the
`async` calls run without needing a separate event loop.

In [ ]:
# Uncomment to run the end-to-end demo (requires API key if applicable).
# await main()